# FineWeb-Edu GPT

Pretraining a decoder-only Transformer (grouped-query attention + RoPE + QK-norm) on [FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) `sample-10BT`, tokenized with the pretrained [LiquidAI/LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M) subword tokenizer.

Unlike text8 (one continuous stream), FineWeb-Edu is many independent web documents, so the datamodule appends an end-of-document token (`<|endoftext|>`) after each one. The optimizer is **Muon for the hidden weight matrices + AdamW for the embeddings / head / norms**.

In [1]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina', 'svg']
%load_ext autoreload
%autoreload 2

import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
from lightning import Trainer, seed_everything
from torchinfo import summary

from chimera.data import FineWebEduDataModule
from chimera.models import GPT
from chimera.modules import LanguageModelModule
from chimera.optim import LinearWarmupCosineAnnealingLR, Muon, muon_param_groups

# datasets + tokenizer caches live on the big volume
os.environ["HF_HOME"] = "/mnt/ai/data/hf"
os.environ["DATA_DIR"] = "/mnt/ai/data"

# Reproducibility: seed all RNGs (incl. dataloader workers). Pair with
# Trainer(deterministic=True) below for deterministic CUDA kernels too.
SEED = 42
seed_everything(SEED, workers=True)

SEQ_LEN = 2048

Seed set to 42


## Data

Load FineWeb-Edu `sample-10BT` and tokenize it with the fixed **LiquidAI/LFM2.5-230M** subword tokenizer (vocab ≈ 64k). Documents are concatenated with `<|endoftext|>` appended after each, so the model learns document boundaries instead of attending across unrelated pages.

`max_train_tokens` caps how many tokens are materialised into memory — the full 10BT sample is ~10B tokens, far more than fits in RAM with this simple in-memory pipeline. Raise it (or move to an on-disk memmap) for a real run.

In [2]:
dm = FineWebEduDataModule(
    data_dir=os.environ["DATA_DIR"],
    name="sample-10BT",
    batch_size=32,
    seq_len=SEQ_LEN,
    tokenizer_backend="pretrained",  # LiquidAI/LFM2.5-230M
    add_eos=True,  # append <|endoftext|> after each document
    max_train_tokens=1_000_000_000,
    num_workers=4,
)
dm.prepare_data()
dm.setup("fit")

train_loader = dm.train_dataloader()
val_loader = dm.val_dataloader()

print(f"tokenizer={dm.pretrained_id}  vocab_size={dm.vocab_size}")
print(f"eos_token={dm.eos_token!r}  eos_id={dm.eos_id}")
print(f"train chunks={len(dm.train_dataset):,}  val chunks={len(dm.val_dataset):,}")

tokenizer=LiquidAI/LFM2.5-230M  vocab_size=64402
eos_token='<|endoftext|>'  eos_id=2
train chunks=483,398  val chunks=4,882


In [ ]:
x, y = next(iter(train_loader))
print("sample:", repr(dm.decode(x[0][:64])))

# confirm the document separator is actually present in the stream
n_eos = int((x == dm.eos_id).sum())
print(f"<|endoftext|> tokens in one batch of {x.numel():,}: {n_eos}")

# most frequent tokens in one batch, shown as their decoded text
counts = Counter(x.flatten().tolist())
top = counts.most_common(20)
labels = [repr(dm.decode([tid])) for tid, _ in top]
values = [c for _, c in top]

plt.figure(figsize=(12, 4))
plt.bar(range(len(top)), values)
plt.title("Top-20 Tokens (one training batch)")
plt.xlabel("Token")
plt.ylabel("Count")
plt.xticks(range(len(top)), labels, rotation=60, ha="right")
plt.tight_layout()
plt.show()

## Model

A decoder-only Transformer with grouped-query attention, rotary position embeddings, and QK-norm. `block_size` must match the datamodule's `seq_len`.

In [ ]:
model = GPT(
    vocab_size=dm.vocab_size,
    block_size=SEQ_LEN,
    n_embd=384,
    n_head=12,
    n_kv_head=3,
    n_layer=6,
    tie_embedding=True,
)
summary(
    model,
    input_data=torch.zeros(1, SEQ_LEN, dtype=torch.long),
    col_names=["output_size", "num_params"],
    depth=1,
)

## Training

Optimize with **Muon + AdamW**: `muon_param_groups` routes the 2D hidden weight matrices (attention + MLP) to Muon and the token embedding, output head, biases, and norm gains to AdamW. Both groups share one `LinearWarmupCosineAnnealingLR` schedule (each anneals from its own base LR).

`N_EPOCH=1` over the capped token budget is a quick smoke run — raise the token cap and epochs for a real one.

In [ ]:
N_EPOCH = 1

optimizer = Muon(
    muon_param_groups(
        model,
        muon_lr=0.02,
        adamw_lr=8e-4,
        adamw_weight_decay=0.01,
    )
)
scheduler = LinearWarmupCosineAnnealingLR(
    optimizer,
    warmup_steps=100,
    n_epochs=N_EPOCH,
    train_loader=train_loader,
)

model = torch.compile(model, mode="reduce-overhead")
# use_cce: fuse lm_head + cross-entropy (apple/ml-cross-entropy), no logits materialized
lm_module = LanguageModelModule(model, optimizer, scheduler, use_cce=True)

trainer = Trainer(
    max_epochs=N_EPOCH,
    precision="bf16-true",
    gradient_clip_val=1.0,
    deterministic=True,
)

trainer.fit(lm_module, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(lm_module, dataloaders=val_loader)

## Analysis

Plot the logged loss and bits-per-token curves, then sample text from the trained model.

In [ ]:
metrics = np.genfromtxt(
    f"{trainer.logger.log_dir}/metrics.csv", delimiter=",", names=True
)


def series(step_key, value_key):
    step = metrics[step_key]
    value = metrics[value_key]
    mask = ~np.isnan(value)
    return step[mask], value[mask]


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Training Metrics")

for ax, metric, title in zip(
    [axes[0], axes[1]], ["loss", "bpt"], ["Loss (nats)", "Bits per Token"]
):
    train_step, train_val = series("step", f"train{metric}")
    val_step, val_val = series("step", f"val{metric}")
    ax.plot(train_step, train_val, marker="o", label="train")
    ax.plot(val_step, val_val, marker="o", label="val")
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.show()

In [ ]:
# Lightning moves the model to CPU after trainer.test(), so put it back on the GPU —
# otherwise generation runs on CPU and is ~30x slower.
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

prompt = "The role of education in modern society is "
idx = torch.tensor([dm.tokenizer.encode(prompt)], device=device)

# Eager by default (no compile warmup). Pass compile=True for ~2x faster decode
# when generating repeatedly — it costs a one-time ~10s compile on the first call.
generated = model.generate(idx, max_new_tokens=200, temperature=0.8)
print(dm.decode(generated[0].cpu()))